In [63]:
# Sorted doubly linked list (largest first)


class Node:
    def __init__(self, i):
        self.id = i
        self.prev = 0
        self.next = 0
        self.val = 0
        self.exists = False

    def __repr__(self):
        return f"{{id: {self.id}, prev: {self.prev}, next: {self.next}, val: {self.val}, exists: {self.exists}}}"


# root = 0
#      <-- prev -- | -- next -->
# root <> head <> ... <> tail <> root
#         max            min
# root.next = head
# root.prev = tail
# head.prev = root
# tail.next = root

# TODO: batch id


class LinkedList:
    def __init__(self):
        # id  => Node
        self.nodes = {}
        self.size = 0

        # Root
        self.nodes[0] = Node(0)

    def get(self, i: int) -> Node:
        return self.nodes.get(i, Node(i))

    def contains(self, i: int) -> bool:
        return self.get(i).exists

    def first(self) -> int:
        root = self.get(0)
        return root.next

    def last(self) -> int:
        root = self.get(0)
        return root.prev

    def prev(self, i: int) -> int:
        node = self.get(i)
        return node.prev

    def next(self, i: int) -> int:
        node = self.get(i)
        return node.next

    def all(self) -> list[Node]:
        nodes = []
        n = self.next(0)
        while n != 0:
            node = self.get(n)
            nodes.append(node)
            n = node.next
        return nodes

    # Descend list (head -> tail) to find a valid insert position
    def desc(self, val: int, start: int) -> (int, int):
        # curent and next
        c = start
        n = self.get(c).next
        while True:
            # next = root or val > next node's val
            if n == 0 or val > self.get(n).val:
                return (c, n)
            else:
                c = n
                n = self.get(c).next

    # Ascend list (head <- tail) to find a valid insert position
    def asc(self, val: int, start: int) -> (int, int):
        # curent and prev
        c = start
        p = self.get(c).prev
        while True:
            # previous = root or val <= previous' val
            if p == 0 or val <= self.get(p).val:
                return (p, c)
            else:
                c = p
                p = self.get(c).prev

    def desc_and_asc(self, val: int, desc_start: int, asc_start: int) -> (int, int):
        dc = desc_start
        dn = self.get(dc).next
        ac = asc_start
        ap = self.get(ac).prev

        while True:
            # Descend 1 node
            if dn == 0 or val > self.get(dn).val:
                return (dc, dn)
            else:
                dc = dn
                dn = self.get(dc).next
            # Ascend 1 node
            if ap == 0 or val <= self.get(ap).val:
                return (ap, ac)
            else:
                ac = ap
                ap = self.get(ac).prev

    def find_insert_pos(self, val: int, p: int, n: int) -> (int, int):
        # prev val >= val > next val
        if p == 0:
            return self.desc(val, 0)
        else:
            # Invalid p or val > prev val
            if not self.contains(p) or self.get(p).val < val:
                p = 0
        if n == 0:
            return self.asc(val, 0)
        else:
            # Invalid n or val < next val
            if not self.contains(n) or val < self.get(n).val:
                n = 0

        if p == 0 and n == 0:
            return self.desc(val, 0)
        elif p == 0:
            return self.asc(val, n)
        elif n == 0:
            return self.desc(val, p)
        else:
            return self.desc_and_asc(val, p, n)

    def valid_insert_pos(self, val: int, p: int, n: int) -> bool:
        if self.get(p).next != n or self.get(n).prev != p:
            return False
        if p != 0 and self.get(p).val < val:
            return False
        if n != 0 and val <= self.get(n).val:
            return False
        return True

    def insert(self, i: int, val: int, p: int, n: int):
        assert not self.contains(i), "already inserted"
        assert i != 0, "cannot insert into root node"
        assert val > 0, "val = 0"

        if not self.valid_insert_pos(val, p, n):
            (p, n) = self.find_insert_pos(val, p, n)

        # Initialize nodes
        self.nodes[i] = self.get(i)
        self.nodes[p] = self.get(p)
        self.nodes[n] = self.get(n)

        # Update
        self.nodes[i].val = val

        assert self.get(p).next == n
        assert self.get(n).prev == p

        self.nodes[p].next = i
        self.nodes[i].prev = p
        self.nodes[i].next = n
        self.nodes[n].prev = i

        self.nodes[i].exists = True
        self.size += 1

    def remove(self, i):
        assert self.contains(i), "not inserted"
        assert i != 0, "cannot remove root node"

        node = self.nodes[i]
        p = node.prev
        n = node.next
        
        assert self.nodes[p].next == i
        assert self.nodes[n].prev == i

        self.nodes[p].next = n
        self.nodes[n].prev = p
        
        del self.nodes[i]
        self.size -= 1


linked_list = LinkedList()
for i in range(1, 10):
    linked_list.insert(i, i, 0, 0)

linked_list.remove(5)

print("root", linked_list.get(0))
print("--- all ---")
linked_list.all()

root {id: 0, prev: 1, next: 9, val: 0, exists: False}
--- all ---


[{id: 9, prev: 0, next: 8, val: 9, exists: True},
 {id: 8, prev: 9, next: 7, val: 8, exists: True},
 {id: 7, prev: 8, next: 6, val: 7, exists: True},
 {id: 6, prev: 7, next: 4, val: 6, exists: True},
 {id: 4, prev: 6, next: 3, val: 4, exists: True},
 {id: 3, prev: 4, next: 2, val: 3, exists: True},
 {id: 2, prev: 3, next: 1, val: 2, exists: True},
 {id: 1, prev: 2, next: 0, val: 1, exists: True}]